In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor, AutoTokenizer
from qwen_vl_utils import process_vision_info
from dots_ocr.utils import dict_promptmode_to_prompt

model_path = "./weights/DotsOCR"
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    attn_implementation="flash_attention_2",
    torch_dtype=torch.bfloat16,
    device_map="cuda:0",
    trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)

/home/zechuan/miniconda3/envs/finance_RAG/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/zechuan/miniconda3/envs/finance_RAG/lib/python3.11/site-packages/transformers/utils/hub.py:105: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.17s/it]
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [2]:
from pdf2image import convert_from_path
import os

# Test PDF to image conversion
pdf_path = "/mnt/SSD2_8TB/zechuan/m3docrag/contents/2024_Tencent_ESG.pdf"
save_dir = "test_images"
os.makedirs(save_dir, exist_ok=True)
if os.path.exists(pdf_path):
    pages = convert_from_path(pdf_path, dpi=200)
    print(f"Converted {len(pages)} pages")
    # Save first page as test image
    if pages:
        pages[54].save(f"{save_dir}/test_page_55.png", "PNG")
        print("Saved page 55 as test_page_55.png")
else:
    print("PDF file not found")


Converted 111 pages
Saved page 55 as test_page_55.png


In [2]:
image_path = "test_images/test_page_55.png"
prompt = """Please output the layout information from the PDF image, including each layout element's bbox, its category, and the corresponding text content within the bbox.

1. Bbox format: [x1, y1, x2, y2]

2. Layout Categories: The possible categories are ['Caption', 'Footnote', 'Formula', 'List-item', 'Page-footer', 'Page-header', 'Picture', 'Section-header', 'Table', 'Text', 'Title'].

3. Text Extraction & Formatting Rules:
    - Picture: For the 'Picture' category, the text field should be omitted.
    - Formula: Format its text as LaTeX.
    - Table: Format its text as HTML.
    - All Others (Text, Title, etc.): Format their text as Markdown.

4. Constraints:
    - The output text must be the original text from the image, with no translation.
    - All layout elements must be sorted according to human reading order.

5. Final Output: The entire output must be a single JSON object.
"""

messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image_path
                },
                {"type": "text", "text": prompt}
            ]
        }
    ]

# Preparation for inference
text = processor.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)


In [3]:
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)

inputs = inputs.to("cuda:0")

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=24000)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
/home/zechuan/miniconda3/envs/finance_RAG/lib/python3.11/site-packages/transformers/utils/generic.py:965: FutureWarning: Both `num_logits_to_keep` and `logits_to_keep` are set for `DotsOCRForCausalLM.forward`. Using `logits_to_keep=1` and ignoring deprecated `num_logits_to_keep=None`.
  output = func(self, *args, **kwargs)


['[{"bbox": [130, 75, 300, 128], "category": "Page-header", "text": "导言"}, {"bbox": [392, 81, 550, 128], "category": "Page-header", "text": "企业管治"}, {"bbox": [620, 81, 770, 128], "category": "Page-header", "text": "ESG管治"}, {"bbox": [847, 81, 1040, 128], "category": "Page-header", "text": "1. 保护环境"}, {"bbox": [1114, 81, 1377, 128], "category": "Page-header", "text": "2. 关心员工成长"}, {"bbox": [1447, 81, 1715, 128], "category": "Page-header", "text": "3. 保障数字权益"}, {"bbox": [1787, 81, 1987, 128], "category": "Page-header", "text": "4. 数字包容"}, {"bbox": [2064, 81, 2566, 128], "category": "Page-header", "text": "5. 数字技术助力可持续发展目标"}, {"bbox": [2646, 81, 2850, 128], "category": "Page-header", "text": "6. 商业道德"}, {"bbox": [2918, 81, 3005, 128], "category": "Page-header", "text": "附录"}, {"bbox": [130, 351, 374, 422], "category": "Section-header", "text": "# 安全运营"}, {"bbox": [130, 473, 2305, 522], "category": "Text", "text": "腾讯持续整合安全能力和资源,优化网络安全风险的发现和修复流程,实现风险管理闭环、数据可视化、自动化修复及验证等一系列功能。"}, {"bbox": [